### Data Preprocessing and Feature Engineering (Exploratory Data Analysis)

Full List of Possible things to see in the Data:

ML industry course

1. new categories appear at testing time, how will the model know how to establish a relationship between the value of the feature and the actual target

2. two features might be perfectly correlated or anti-correlated, knowing one feature might perfectly inform you on the value of another, no need for both

3. Duplicates in the dataframe: we always need to have an ID (for classification/regression it's usually one key, for time series is an id and a time index)

4. perfect correlation in terms of whether the feature is present or not 

5. Missing values, completely at random, at random, not at random

6. Examples of features that are leaking the target, might not be present at production time

7.	Inconsistent categorical labels
Same concept written differently (e.g. “NY”, “New York”, “new york”). Very common in operational data.

8.	Implicit hierarchies inside categories
Values that belong to a taxonomy (e.g. product category → subcategory) but stored as flat strings.

9.	Unit inconsistencies
Same feature stored in different units (€, $, thousands vs units, meters vs km).

10.	Data drift across time
Distribution of a feature changes by period (e.g. pre/post policy change). Even visible just with simple plots.

11.	Rare categories / long tail
Many categories with very few observations.

12.	Mixed types inside a column
Numeric stored as strings, free text mixed with codes.

13.	Invalid or impossible values
Negative ages, future dates, zero where impossible.

14.	Truncated / capped values
Operational limits (e.g. max credit limit stored as 999999).

15.	Sampling bias / selection bias
Dataset represents only approved customers, only active users, etc.

16.	Multiple rows per entity when you expect one
Hidden panel structure.

17.	Time misalignment
Features measured after the event you want to predict.

18.	Sparse indicator explosions
Hundreds of mostly-zero columns from upstream systems.

19.	Business process artifacts
Values driven by workflow rules rather than real behavior (e.g. default flags set after manual review).

In [1]:
cd ..

/Users/tommasoguerrini/Projects/ml_industry_course


### Datasets

In [2]:
import tabulate
import pandas as pd
pd.options.display.max_columns = 200

def show(df, n_rows=5):
    print(tabulate.tabulate(df.head(n_rows), headers='keys', tablefmt='psql'))

In [45]:
import os

### Dataset Overview



| Dataset | Task | Clean file (rows x cols) | Issues file (rows x cols) | Main injected issues |
|---|---|---:|---:|---|
| Adult Income | Binary classification (`class`: `<=50K` / `>50K`) | `adult_income_clean.csv` (48,842 x 17) | `adult_income_issues.csv` (50,317 x 36) | Missingness, leakage, duplicates, mixed types, invalid values, rare/new categories, sparse indicators, process artifacts |
| Ames Housing | Regression (`sale_price`) | `ames_housing_clean.csv` (1,460 x 82) | `ames_housing_issues.csv` (1,576 x 84) | Truncation/capping, multiple rows per entity |
| Retail Panel | Time-aware store-level forecasting (`sales`) | `retail_panel_clean.csv` (22,800 x 6) | `retail_panel_issues.csv` (22,800 x 9) | Data drift, time misalignment, process artifacts |

### 1) Adult Income (`adult_income_issues.csv`)
Person-level application/income dataset with added operational and quality issues.

| person_id | age | occupation | hours_per_week | native_country | class | split | post_adjudication_risk_code | duplicate_application_flag | record_written_at |
|---|---:|---|---:|---|---|---|---:|---:|---|
| 1 | 25 | Machine-op-inspct | 40 | United-States | <=50K | train | 0.005 | 0 | 2026-01-20 03:21:27 |
| 2 | 38 | Farming-fishing | 50 | United-States | <=50K | train | 0.021 | 0 | 2026-02-08 22:54:51 |
| 3 | 28 | Protective-serv | 40 | US | >50K | test | 0.992 | 0 | 2026-01-23 11:09:29 |

### 2) Ames Housing (`ames_housing_issues.csv`)
Property-level housing dataset for sale price prediction, with panel-style duplicates and capped derived values.

| property_id | mszoning | neighborhood | lotarea | grlivarea | sale_price | reported_max_loan | record_extract_month |
|---|---|---|---:|---:|---:|---:|---|
| 1 | RL | CollgCr | 8450 | 1710 | 208500.0 | 187650.0 | 2024-01 |
| 2 | RL | Veenker | 9600 | 1262 | 181500.0 | 163350.0 | 2024-01 |
| 3 | RL | CollgCr | 11250 | 1786 | 223500.0 | 201150.0 | 2024-01 |

### 3) Retail Panel (`retail_panel_issues.csv`)
Daily store panel with promotion/inventory context and injected temporal/business-process issues.

| store_id | date | inventory_units | promotion_discount_pct | sales | inventory_after_restock | manual_override_flag | workflow_route_code |
|---:|---|---:|---:|---:|---:|---:|---|
| 1 | 2023-01-01 | 561 | 7.59 | 143 | 702.0 | 0 | AUTO_PASS |
| 1 | 2023-01-02 | 702 | 11.79 | 139 | 423.0 | 0 | AUTO_PASS |
| 1 | 2023-01-03 | 423 | 1.43 | 88 | 448.0 | 0 | AUTO_PASS |


In [38]:
adult = pd.read_csv("day1/generated/adult_income_issues.csv")
ames = pd.read_csv("day1/generated/ames_housing_issues.csv")
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")

Duplicates can artificially inflate sample size, bias learned patterns, and create train/test contamination if the same entity appears in both sets.



In [42]:
print("Adult exact duplicate rows:", adult.duplicated().sum())
print("Ames exact duplicate rows:", ames.duplicated().sum())
print("Retail exact duplicate rows:", retail.duplicated().sum())


# B) Duplicate entity keys (classification/regression)
print("Adult duplicate person_id rows:", adult.duplicated(subset=["person_id"], keep=False).sum())
print("Ames duplicate property_id rows:", ames.duplicated(subset=["property_id"], keep=False).sum())

print(
    "Retail duplicate (store_id, date) rows:",
    retail.duplicated(subset=["store_id", "date"], keep=False).sum()
)

if "split" in adult.columns:
    duplicated_entities = adult[adult.duplicated(subset=["person_id"], keep=False)]
    cross_split = duplicated_entities.groupby("person_id")["split"].nunique().gt(1).sum()
    print("Duplicated person_id appearing in >1 split:", cross_split)

Adult exact duplicate rows: 596
Ames exact duplicate rows: 0
Retail exact duplicate rows: 0
Adult duplicate person_id rows: 2931
Ames duplicate property_id rows: 232
Retail duplicate (store_id, date) rows: 0
Duplicated person_id appearing in >1 split: 0
